# Custom dataset collation

In this notebook, we'll create a custom collation function for the HyraxRandomDataset.

In general custom collation functions are only required when the input data does not have a uniform shape.
For instance, a selection of stars generally wouldn't all have the same number of photometric measurements over time.
To provide this data to a model, the samples with fewer photometric observations would be padded with placeholder values
such that all the data samples would have the same shape.
A mask would be defined that would indicate which values are placeholders.
The model that consumes this data would need to appropriate handle the padded data using the information in the mask.

For simplicity here, we'll create a collation function for a dataset that has a uniform shape.
This allows us to focus on the mechanism, instead of the logic to collate a particular dataset.
We will still create a mask, but it's values will all be `False`.

As usual, we'll begin by creating an instance of Hyrax

In [ ]:
from hyrax import Hyrax

h = Hyrax()

Next we'll create a data request

In [ ]:
data_request = {
    "train": {
        "data": {
            "dataset_class": "HyraxRandomDataset",
            "data_location": "./data",
            "fields": ["object_id", "image", "label"],
            "split_fraction": 1.0,
            "primary_id_field": "object_id",
        },
    }
}

h.set_config("data_request", data_request)

## Add a custom collation function

Here we'll dynamically add a collation function to the HyraxRandomDataset class.
Typically, this would be in the file where the class is defined.
It can be added in a notebook, as shown, for faster experimentation.

In [ ]:
from hyrax.datasets.random.hyrax_random_dataset import HyraxRandomDataset
import numpy as np


@staticmethod
def collate(samples: list[dict]) -> dict:
    """Collate a list of dictionaries into a single batch.

    This method takes a list of samples and collates them into a single batch.
    The returned batch dictionary will contain the following keys:

    - ``object_id``: Numpy array of object IDs for the samples in the batch.
    - ``image``: Numpy array of stacked images for the samples in the batch.
    - ``mask``: Numpy array of masks with the same shape as the images.
    - ``label``: Numpy array of labels for the samples in the batch (if provided).

    Parameters
    ----------
    samples : list of dict
        A list of samples to collate.

    Returns
    -------
    dict
        A single dictionary containing the collated data as numpy arrays.
    """
    collated_data = {
        "object_id": np.array([sample["object_id"] for sample in samples]),
        # vertically stack all the images into a single NumPy array
        "image": np.stack([sample["image"] for sample in samples], axis=0),
    }

    # create a "mask" that has the same shape as the "image" stack.
    collated_data["mask"] = np.zeros_like(collated_data["image"], dtype=bool)

    if "label" in samples[0]:
        collated_data["label"] = np.array([sample["label"] for sample in samples])

    return collated_data


# Attach the collation function to the HyraxRandomDataset class
HyraxRandomDataset.collate = collate

## Initialize the data request

Calling `h.prepare()` will instantiate the dataset class and allow us to request data samples by index.

In [ ]:
dataset = h.prepare()

# Access the training dataset for clearity in the next steps
train_dataset = dataset["train"]

## Request samples

Here we request a couple of samples from the dataset by index, and then print out some metadata.

In [ ]:
sample_0 = train_dataset[0]
sample_1 = train_dataset[1]

In [ ]:
print("Type and shape / value of fields in sample 0:")
print(f"object_id type: {type(sample_0['data']['object_id'])}, value: {sample_0['data']['object_id']}")
print(f"image type: {type(sample_0['data']['image'])}, shape: {sample_0['data']['image'].shape}")
print(f"label type: {type(sample_0['data']['label'])}, value: {sample_0['data']['label']}")

## Collate two samples

Let's collate the two samples that were selected and examine the resulting types and shapes.

In [ ]:
collated = train_dataset.collate([sample_0, sample_1])

In [ ]:
print("Collated types and shapes from samples 0 and 1:")
print(f"image type: {type(collated['data']['image'])}, shape: {collated['data']['image'].shape}")
print(f"object_id type: {type(collated['data']['object_id'])}, shape: {collated['data']['object_id'].shape}")
print(f"label type: {type(collated['data']['label'])}, shape: {collated['data']['label'].shape}")
print(f"mask type: {type(collated['data']['mask'])}, shape: {collated['data']['mask'].shape}")

Note that there is now a `mask` data field that has the same type and shape as `image`.
